In [2]:
import os, random
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from difflib import SequenceMatcher
import zipfile


In [4]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/recognition.zip'
extract_path = '/content/recognition_dataset'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(" Files extracted to:", extract_path)
print(os.listdir(os.path.join(extract_path, "recognition")))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Files extracted to: /content/recognition_dataset
['test', 'test.csv', 'train', 'train.csv']


In [5]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
import torch.nn as nn

PAD_TOKEN, SOS_TOKEN, EOS_TOKEN = "<PAD>", "<SOS>", "<EOS>"

class OCRDataset(Dataset):
    def __init__(self, csv_path, base_folder, transform=None, vocab=None):
        self.df = pd.read_csv(csv_path)
        self.base_folder = base_folder
        self.transform = transform

        if vocab is None:
            self.char2idx = self._build_vocab(self.df["Text"])
            self.idx2char = {v: k for k, v in self.char2idx.items()}
        else:
            self.char2idx, self.idx2char = vocab

    def _build_vocab(self, texts):
        chars = set("".join(map(str, texts)))
        vocab = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2}
        for i, c in enumerate(sorted(chars)):
            vocab[c] = i + 3
        return vocab

    def text_to_seq(self, text):
        return [self.char2idx[SOS_TOKEN]] + [self.char2idx[c] for c in str(text)] + [self.char2idx[EOS_TOKEN]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.base_folder, row["Filepath"])
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)
        seq = torch.tensor(self.text_to_seq(row["Text"]), dtype=torch.long)
        return image, seq


def collate_fn(batch):
    imgs, seqs = zip(*batch)
    imgs = torch.stack(imgs)
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return imgs, padded


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# ---------------- CNN Encoder ----------------
class CNNEncoder(nn.Module):
    def __init__(self, in_channels=1, feat_dim=512):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels,64,3,1,1), nn.ReLU(True), nn.BatchNorm2d(64),
            nn.Conv2d(64,64,3,1,1), nn.ReLU(True), nn.BatchNorm2d(64),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,1,1), nn.ReLU(True), nn.BatchNorm2d(128),
            nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,1,1), nn.ReLU(True), nn.BatchNorm2d(256),
            nn.MaxPool2d((2,2),(2,1),(0,1)),
            nn.Conv2d(256,512,3,1,1), nn.ReLU(True), nn.BatchNorm2d(512),
            nn.MaxPool2d((2,2),(2,1),(0,1)),
            nn.Conv2d(512,feat_dim,2,1,0), nn.ReLU(True)
        )
        self.proj = nn.Linear(1536, feat_dim)


    def forward(self, x):
        x = self.cnn(x)
        B, C, H, W = x.size()
        x = x.permute(0, 3, 1, 2).reshape(B, W, C * H)
        return self.proj(x)

# ---------------- Attention ----------------
class Attention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim + dec_dim, dec_dim)
        self.v = nn.Linear(dec_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        B, T, _ = encoder_outputs.size()
        hidden = hidden.unsqueeze(1).repeat(1, T, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attn_weights = torch.softmax(self.v(energy).squeeze(2), dim=1)
        return attn_weights

# ---------------- Decoder with Attention ----------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_dim, dec_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim + enc_dim, dec_dim, batch_first=True)
        self.fc = nn.Linear(dec_dim, vocab_size)
        self.attention = Attention(enc_dim, dec_dim)

    def forward(self, encoder_out, target_seq, teacher_forcing_ratio=0.5):
        B = encoder_out.size(0)
        hidden = torch.zeros(1, B, self.lstm.hidden_size).to(encoder_out.device)
        cell = torch.zeros(1, B, self.lstm.hidden_size).to(encoder_out.device)
        seq_len = target_seq.size(1)
        outputs = torch.zeros(B, seq_len, self.fc.out_features).to(encoder_out.device)
        input_token = target_seq[:, 0]

        for t in range(1, seq_len):
            emb = self.embedding(input_token)
            attn_weights = self.attention(hidden[-1], encoder_out)
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_out).squeeze(1)
            lstm_input = torch.cat((emb, context), dim=1).unsqueeze(1)
            out, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
            out = self.fc(out.squeeze(1))
            outputs[:, t] = out
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = target_seq[:, t] if teacher_force else out.argmax(1)
        return outputs


In [14]:
import torch
import torch.nn.functional as F

def beam_search_decode(encoder, decoder, image, sos_idx, eos_idx, beam_width=3, max_len=30):
    encoder_out = encoder(image.unsqueeze(0))
    device = image.device

    hidden = torch.zeros(1, 1, decoder.lstm.hidden_size).to(device)
    cell = torch.zeros(1, 1, decoder.lstm.hidden_size).to(device)

    sequences = [[list([sos_idx]), 0.0, hidden, cell]]

    for _ in range(max_len):
        all_candidates = []
        for seq, score, hidden, cell in sequences:
            if seq[-1] == eos_idx:
                all_candidates.append([seq, score, hidden, cell])
                continue

            input_token = torch.tensor([seq[-1]]).to(device)
            emb = decoder.embedding(input_token)
            attn_weights = decoder.attention(hidden[-1], encoder_out)
            context = torch.bmm(attn_weights.unsqueeze(1), encoder_out).squeeze(1)
            lstm_input = torch.cat((emb, context), dim=1).unsqueeze(1)
            out, (hidden, cell) = decoder.lstm(lstm_input, (hidden, cell))
            logits = decoder.fc(out.squeeze(1))
            log_probs = F.log_softmax(logits, dim=1)
            topk_probs, topk_idx = log_probs.topk(beam_width)

            for i in range(beam_width):
                candidate = [seq + [topk_idx[0][i].item()],
                             score + topk_probs[0][i].item(),
                             hidden, cell]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates, key=lambda tup: tup[1], reverse=True)
        sequences = ordered[:beam_width]

    return sequences[0][0]  # return best sequence


In [ ]:


import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import os
from difflib import SequenceMatcher


def cer(pred_seq, true_seq, inv_vocab):
    pred_text = "".join([inv_vocab[i] for i in pred_seq if i not in [0,1,2]])
    true_text = "".join([inv_vocab[i] for i in true_seq if i not in [0,1,2]])
    return 1 - SequenceMatcher(None, pred_text, true_text).ratio()

def word_acc(pred_seq, true_seq, inv_vocab):
    pred_text = "".join([inv_vocab[i] for i in pred_seq if i not in [0,1,2]])
    true_text = "".join([inv_vocab[i] for i in true_seq if i not in [0,1,2]])
    return pred_text == true_text

# ---------------- Paths ----------------
base_dir = "/content/recognition_dataset/recognition"
train_csv = os.path.join(base_dir, "train.csv")
test_csv = os.path.join(base_dir, "test.csv")

transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor()
])

train_dataset = OCRDataset(train_csv, base_dir, transform=transform)
test_dataset = OCRDataset(test_csv, base_dir, transform=transform,
                          vocab=(train_dataset.char2idx, train_dataset.idx2char))

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# ---------------- Model, Loss, Optimizer ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = CNNEncoder().to(device)
decoder = Decoder(len(train_dataset.char2idx), emb_dim=128, enc_dim=512, dec_dim=512).to(device)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# ---------------- Training ----------------
best_acc = 0
epochs = 10
inv_vocab = train_dataset.idx2char

for epoch in range(1, epochs+1):
    encoder.train()
    decoder.train()
    total_loss = 0

    for imgs, seqs in train_loader:
        imgs, seqs = imgs.to(device), seqs.to(device)
        optimizer.zero_grad()
        enc_out = encoder(imgs)
        outputs = decoder(enc_out, seqs)
        loss = criterion(outputs[:,1:].reshape(-1, outputs.size(-1)), seqs[:,1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    if epoch % 5 == 0:
        encoder.eval(); decoder.eval()
        total_cer, total_acc, count = 0, 0, 0
        with torch.no_grad():
            for imgs, seqs in test_loader:
                imgs, seqs = imgs.to(device), seqs.to(device)
                for i in range(imgs.size(0)):
                    pred_seq = beam_search_decode(encoder, decoder, imgs[i], 1, 2, beam_width=3)
                    true_seq = seqs[i].cpu().tolist()
                    total_cer += cer(pred_seq, true_seq, inv_vocab)
                    total_acc += int(word_acc(pred_seq, true_seq, inv_vocab))
                    count += 1
                break

        cer_score = total_cer/count
        acc = total_acc/count*100
        print(f" → Epoch {epoch}: CER={cer_score:.3f}, Word Acc={acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            torch.save({'encoder': encoder.state_dict(),
                        'decoder': decoder.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'epoch': epoch}, "best_model.pth")
            print(f"New Best Model Saved ({acc:.2f}%)!\n")


Epoch 1/10 - Loss: 5.1440
Epoch 2/10 - Loss: 5.0888
Epoch 3/10 - Loss: 5.0740
Epoch 4/10 - Loss: 5.0662
Epoch 5/10 - Loss: 5.0629
 → Epoch 5: CER=1.000, Word Acc=0.00%
Epoch 6/10 - Loss: 5.0593
Epoch 7/10 - Loss: 5.0591
Epoch 8/10 - Loss: 5.0567
Epoch 9/10 - Loss: 5.0545
Epoch 10/10 - Loss: 5.0534
 → Epoch 10: CER=0.986, Word Acc=0.00%


In [ ]:
import torch
import torchvision.transforms as transforms
import os

torch.save({
    'encoder': encoder.state_dict(),
    'decoder': decoder.state_dict(),
    'optimizer': optimizer.state_dict(),
    'epoch': epoch
}, "best_model.pth")

base_dir = "/content/recognition_dataset/recognition"
train_csv = os.path.join(base_dir, "train.csv")
img_path = "/content/recognition_dataset/recognition/test/tamil/D_image_635_5.jpg"  # single image path

# ---------------- Transform ----------------
transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor()
])

# ---------------- Load vocab ----------------
dummy_dataset = OCRDataset(train_csv, base_dir, transform=transform)
idx2char = dummy_dataset.idx2char
sos_idx = dummy_dataset.char2idx["<SOS>"]
eos_idx = dummy_dataset.char2idx["<EOS>"]
vocab_size = len(dummy_dataset.char2idx)

# ---------------- Load model ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = CNNEncoder().to(device)
decoder = Decoder(vocab_size=vocab_size, emb_dim=128, enc_dim=512, dec_dim=512).to(device)

checkpoint = torch.load("best_model.pth", map_location=device)
encoder.load_state_dict(checkpoint['encoder'])
decoder.load_state_dict(checkpoint['decoder'])
encoder.eval()
decoder.eval()

# ---------------- Load and preprocess image ----------------
image = Image.open(img_path).convert("L")
image = transform(image).to(device)

# ---------------- Beam search prediction ----------------
pred_seq = beam_search_decode(encoder, decoder, image, sos_idx, eos_idx, beam_width=5, max_len=50)

# ---------------- Decode sequence to text ----------------
pred_text = "".join([idx2char[idx] for idx in pred_seq if idx not in [0,1,2]])
print("Predicted Text:", pred_text)


Predicted Text: ஈஈൺ॰ಥंಥंکଳூூଡ଼ଡ଼KഴఆॱउഴॱrதL੦ਲ਼ꯑꯑ]ǝरररररડડಓಓಓಓಓಓಓறఽ૧૧౯౯
